In [47]:
import pandas as pd
import json
import logging
from datetime import datetime 
import os

logging.basicConfig(
    level=logging.INFO , 
    format= "%(asctime)s-%(levelname)s-%(message)s")

#---------helper functions---------------------------------------

def product_score(row):
    score=0
    if row['Sales'] >1000 : score+=30
    if row['Quantity'] > 5 : score+=20
    if row['Discount']<0.15 : score+=25
    if row['Profit'] > 200 : score+=25
    return score

def discount_risk(x):
    if x>0.30 : return 'High Risk'
    elif x>=0.15 : return 'Medium Risk'
    else:return 'Safe'

def profit_category(row):
    if row['Sales'] == 0 : return 'unknown'
    margin = (row['Profit']/row['Sales'])*100
    if margin >20: return'High Profit'
    elif margin >= 0 : return "Break even"
    else:return "Loss"

#-------pipeline class ------------------------------------------

class Datapipeline:
    
    def __init__(self, file_path):
        self.file_path= file_path
        self.df= None
        self.report = {}

    def load_file(self):
        self.df = pd.read_csv(self.file_path , encoding='latin1')
        return self

    def validation(self):
            required = ['Sales' , 'Discount' , 'Profit', 'Quantity']
            missing = [col for col in required if col not in self.df.columns]
            if missing:
                raise ValueError(f"missing the columns")
            return self 

    def enrich(self):
        self.df['profit_category'] = self.df.apply(profit_category , axis=1)
        self.df['product_score'] = self.df.apply(product_score , axis=1)
        self.df['discount_risk'] = self.df['Discount'].apply(discount_risk)
        return self

    def generate_report(self):
        self.report  = {
            "total_records" : len(self.df),
            "null_count" : self.df.isnull().sum().to_dict(),
            "duplicate":int(self.df.duplicated().sum())
        }
        return self

    def save_report(self):
        os.makedirs("output" , exist_ok=True)
        today= datetime.now().strftime("%Y-%m-%d")
        filename = f"output/report_{today}.json"
        with open(filename ,"w" ) as f:
            json.dump(self.report, f , indent=4)
        return self

    def run(self):
        try:
            return (self
                .load_file()
                .validation()
                .enrich()
                .generate_report()
                .save_report())
        except Exception as e:
            logging.error(f"Pipeline execution has failed {e}")
            return None
            
#-------run----------------------------------------------------

pipeline=Datapipeline("Sample_Superstore.csv")
pipeline.run()
        